# Qiskit Integration

qb-compiler integrates with Qiskit at multiple levels: from a simple
one-liner function to individual passes you can insert into your own
transpiler pipeline.

This notebook covers:
- `qb_transpile()` -- one-liner transpilation with calibration-aware layout
- `QBPassManager.from_backend()` -- full pass manager
- `QBCalibrationLayout` -- standalone Qiskit analysis pass
- Comparing qb-compiler output vs standard Qiskit transpile
- Using qb-compiler passes inside a Qiskit `StagedPassManager`
- Transpiler plugin discovery
- Converting between `QBCircuit` and Qiskit `QuantumCircuit`
- End-to-end workflow: Qiskit circuit -> qb-compiler -> execution

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.compiler import transpile
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qb_compiler.qiskit_plugin import (
    QBCalibrationLayout,
    QBPassManager,
    qb_transpile,
)
from qb_compiler.qiskit_plugin.analysis_passes import (
    QBDepthAnalysis,
    QBGateCountAnalysis,
    QBErrorBudgetAnalysis,
)
from qb_compiler.qiskit_plugin.transformation_passes import (
    QBGateCancellation,
    QBGateDecomposition,
)
from qb_compiler.config import BACKEND_CONFIGS

## 1. The `qb_transpile()` One-Liner

The simplest way to use qb-compiler from Qiskit. It builds a pass
manager with `QBCalibrationLayout` injected and runs it on your circuit.

In [2]:
# Build a test circuit
qc = QuantumCircuit(4)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.cx(2, 3)
qc.measure_all()

print("Original circuit:")
print(qc)

# Transpile with calibration-aware layout
compiled = qb_transpile(
    qc,
    backend="ibm_fez",
    calibration_path="../tests/fixtures/calibration_snapshots/ibm_fez_2026_03_14.json",
    optimization_level=2,
)

print(f"\nCompiled circuit:")
print(f"  Depth: {compiled.depth()}")
print(f"  Gates: {dict(compiled.count_ops())}")

Original circuit:
        ┌───┐                ░ ┌─┐         
   q_0: ┤ H ├──■─────────────░─┤M├─────────
        └───┘┌─┴─┐           ░ └╥┘┌─┐      
   q_1: ─────┤ X ├──■────────░──╫─┤M├──────
             └───┘┌─┴─┐      ░  ║ └╥┘┌─┐   
   q_2: ──────────┤ X ├──■───░──╫──╫─┤M├───
                  └───┘┌─┴─┐ ░  ║  ║ └╥┘┌─┐
   q_3: ───────────────┤ X ├─░──╫──╫──╫─┤M├
                       └───┘ ░  ║  ║  ║ └╥┘
meas: 4/════════════════════════╩══╩══╩══╩═
                                0  1  2  3 


/home/dministrator/miniconda3/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.1) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(



Compiled circuit:
  Depth: 13
  Gates: {'rz': 11, 'sx': 7, 'measure': 4, 'cz': 3, 'barrier': 1}


## 2. QBPassManager.from_backend()

For more control, use `QBPassManager` which creates a full Qiskit
`StagedPassManager` with calibration-aware layout pre-configured.

In [3]:
# Create a pass manager for IBM Fez
pm = QBPassManager.from_backend(
    "ibm_fez",
    calibration_data="../tests/fixtures/calibration_snapshots/ibm_fez_2026_03_14.json",
    optimization_level=2,
)

# Build a more complex circuit
qc = QuantumCircuit(5)
qc.h(0)
for i in range(4):
    qc.cx(i, i + 1)
qc.barrier()
for i in range(5):
    qc.rz(0.5, i)
qc.measure_all()

result = pm.run(qc)
print(f"Original depth: {qc.depth()}")
print(f"Compiled depth: {result.depth()}")
print(f"Compiled gates: {dict(result.count_ops())}")

Original depth: 7
Compiled depth: 17
Compiled gates: {'rz': 19, 'sx': 9, 'measure': 5, 'cz': 4, 'barrier': 2}


## 3. QBCalibrationLayout as a Standalone Pass

`QBCalibrationLayout` is a Qiskit `AnalysisPass` that scores physical
qubits using T1, T2, readout error, and gate error, then assigns
virtual qubits to the best-scoring physical qubits.

You can drop it into *any* existing Qiskit pass manager.

In [4]:
import json

# Load calibration data
with open("../tests/fixtures/calibration_snapshots/ibm_fez_2026_03_14.json") as f:
    cal_data = json.load(f)

# Create the pass
cal_layout = QBCalibrationLayout(cal_data)

# Inject into a standard Qiskit pass manager
spec = BACKEND_CONFIGS["ibm_fez"]
coupling_map = CouplingMap(couplinglist=cal_data["coupling_map"])

pm = generate_preset_pass_manager(
    optimization_level=2,
    basis_gates=list(spec.basis_gates),
    coupling_map=coupling_map,
)
pm.layout.append(cal_layout)

# Run on a test circuit
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.measure_all()

result = pm.run(qc)
print(f"Transpiled with QBCalibrationLayout:")
print(f"  Depth: {result.depth()}")
print(f"  Gates: {dict(result.count_ops())}")

Transpiled with QBCalibrationLayout:
  Depth: 10
  Gates: {'rz': 8, 'sx': 5, 'measure': 3, 'cz': 2, 'barrier': 1}


## 4. Comparing qb_transpile() vs Standard Qiskit transpile()

Let's run the same circuit through both pipelines and compare the
output quality (depth, gate count).

In [5]:
# Build a QAOA-style circuit (common in optimisation)
n_qubits = 6
qc = QuantumCircuit(n_qubits)

# Initial superposition
for i in range(n_qubits):
    qc.h(i)

# Problem layer (ring of ZZ interactions)
for i in range(n_qubits):
    j = (i + 1) % n_qubits
    qc.cx(i, j)
    qc.rz(0.5, j)
    qc.cx(i, j)

# Mixer layer
for i in range(n_qubits):
    qc.rx(0.3, i)

qc.measure_all()

print(f"Original circuit: depth={qc.depth()}, gates={sum(qc.count_ops().values())}")

# Standard Qiskit transpile
spec = BACKEND_CONFIGS["ibm_fez"]
coupling_map = CouplingMap(couplinglist=cal_data["coupling_map"])

qiskit_result = transpile(
    qc,
    basis_gates=list(spec.basis_gates),
    coupling_map=coupling_map,
    optimization_level=2,
    seed_transpiler=42,
)

# qb-compiler transpile
qb_result = qb_transpile(
    qc,
    backend="ibm_fez",
    calibration_data=cal_data,
    optimization_level=2,
)

print(f"\n{'Metric':<20} {'Qiskit':>10} {'qb-compiler':>12}")
print("-" * 44)
print(f"{'Depth':<20} {qiskit_result.depth():>10} {qb_result.depth():>12}")
print(f"{'Total gates':<20} {sum(qiskit_result.count_ops().values()):>10} {sum(qb_result.count_ops().values()):>12}")

qiskit_cx = qiskit_result.count_ops().get('cx', 0)
qb_cx = qb_result.count_ops().get('cx', 0)
print(f"{'CX gates':<20} {qiskit_cx:>10} {qb_cx:>12}")

Original circuit: depth=21, gates=37



Metric                   Qiskit  qb-compiler
--------------------------------------------
Depth                        69           71
Total gates                 135          136
CX gates                      0            0


## 5. Using qb-compiler Passes in a Qiskit StagedPassManager

qb-compiler provides both analysis and transformation passes that
are Qiskit-compatible. You can mix and match them with Qiskit's
built-in passes.

In [6]:
from qiskit.transpiler import PassManager as QiskitPM

# Build a custom analysis pipeline using qb-compiler passes
analysis_pm = QiskitPM([
    QBDepthAnalysis(),
    QBGateCountAnalysis(),
    QBErrorBudgetAnalysis(
        single_qubit_error=0.001,
        two_qubit_error=0.005,
        readout_error=0.01,
    ),
])

# Run analysis on a test circuit
qc = QuantumCircuit(4)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.cx(2, 3)
qc.measure_all()

_ = analysis_pm.run(qc)

# Read results from the property set
ps = analysis_pm.property_set
print(f"Analysis results for GHZ-4 circuit:")
print(f"  Circuit depth:       {ps.get('qb_depth', 'N/A')}")
print(f"  Total gate count:    {ps.get('qb_gate_count', 'N/A')}")
print(f"  2Q gate count:       {ps.get('qb_two_qubit_count', 'N/A')}")
print(f"  Error budget:        {ps.get('qb_error_budget', 'N/A'):.4f}")
print(f"  Estimated fidelity:  {ps.get('qb_estimated_fidelity', 'N/A'):.4f}")
print(f"  Gate counts by type: {ps.get('qb_gate_counts_by_type', {})}")

Analysis results for GHZ-4 circuit:
  Circuit depth:       6
  Total gate count:    9
  2Q gate count:       4
  Error budget:        0.0594
  Estimated fidelity:  0.9406
  Gate counts by type: {'h': 1, 'cx': 3, 'barrier': 1, 'measure': 4}


In [7]:
# qb-compiler transformation passes
# QBGateCancellation removes adjacent self-inverse gate pairs

qc = QuantumCircuit(2)
qc.h(0)
qc.h(0)      # cancels with previous H
qc.cx(0, 1)
qc.cx(0, 1)  # cancels with previous CX
qc.x(0)
qc.measure_all()

print(f"Before cancellation: {sum(qc.count_ops().values())} gates")
print(f"  Gates: {dict(qc.count_ops())}")

cancel_pm = QiskitPM([QBGateCancellation(max_iterations=3)])
cancelled = cancel_pm.run(qc)

print(f"After cancellation:  {sum(cancelled.count_ops().values())} gates")
print(f"  Gates: {dict(cancelled.count_ops())}")

Before cancellation: 8 gates
  Gates: {'h': 2, 'cx': 2, 'measure': 2, 'x': 1, 'barrier': 1}
After cancellation:  4 gates
  Gates: {'measure': 2, 'x': 1, 'barrier': 1}


## 6. Transpiler Plugin Discovery

qb-compiler registers itself as a Qiskit transpiler layout-stage
plugin via `pyproject.toml` entry points. When `qb-compiler` is
installed, Qiskit can discover it automatically and the plugin is
invokable by passing `layout_method="qb_calibration"` to
`generate_preset_pass_manager()`.

```toml
[project.entry-points."qiskit.transpiler.layout"]
qb_calibration = "qb_compiler.qiskit_plugin:QBCalibrationLayoutPlugin"
```

The plugin loads its calibration snapshot from the
`QB_CALIBRATION_PATH` environment variable at plugin-invoke time.
If the variable is unset, it falls back to Qiskit's `TrivialLayout`.

In [8]:
import os

from qiskit.circuit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Point the plugin at a calibration snapshot before building the pass manager.
os.environ["QB_CALIBRATION_PATH"] = "../tests/fixtures/calibration_snapshots/ibm_fez_2026_03_14.json"

# Build a preset pass manager that uses the qb_calibration layout plugin.
circuit = QuantumCircuit(3)
circuit.h(0); circuit.cx(0, 1); circuit.cx(1, 2)
pm = generate_preset_pass_manager(optimization_level=2, layout_method="qb_calibration")
compiled = pm.run(circuit)
print(f"Compiled depth: {compiled.depth()}")

# Verify the plugin is discoverable via Qiskit's entry-point loader.
try:
    import importlib.metadata
    try:
        eps = importlib.metadata.entry_points(group="qiskit.transpiler.layout")
    except TypeError:  # older Python fallback
        eps = importlib.metadata.entry_points().get("qiskit.transpiler.layout", [])
    qb_entries = [ep for ep in eps if ep.name == "qb_calibration"]
    if qb_entries:
        print(f"Discovered qb-compiler layout plugin: {qb_entries[0]}")
    else:
        print("Plugin not found in entry_points (normal if running from source without pip install)")
except Exception as e:
    print(f"Plugin discovery requires pip install: {e}")

# NOTE: the legacy QBTranspilerPlugin().get_pass_manager(calibration_data=...)
# API still works for backward compatibility but emits a DeprecationWarning
# and will be removed in qb-compiler 0.4.0.

Compiled depth: 3
Discovered qb-compiler layout plugin: EntryPoint(name='qb_calibration', value='qb_compiler.qiskit_plugin:QBCalibrationLayoutPlugin', group='qiskit.transpiler.layout')


## 7. Converting Between QBCircuit and Qiskit QuantumCircuit

qb-compiler has its own IR (`QBCircuit`) but provides bidirectional
converters. This lets you use qb-compiler's analysis tools on Qiskit
circuits and vice versa.

In [9]:
from qb_compiler.ir.converters.qiskit_converter import from_qiskit, to_qiskit
from qb_compiler.ir.circuit import QBCircuit as IRCircuit
from qb_compiler.ir.operations import QBGate

# Qiskit -> QBCircuit
qc = QuantumCircuit(3, 3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.barrier()
qc.measure([0, 1, 2], [0, 1, 2])

qb_circ = from_qiskit(qc)
print(f"Qiskit -> QBCircuit:")
print(f"  {qb_circ}")
print(f"  Gates: {qb_circ.gate_count}")
print(f"  2Q gates: {qb_circ.two_qubit_gate_count}")
print(f"  Depth: {qb_circ.depth}")
print(f"  Gate names: {set(g.name for g in qb_circ.gates)}")

Qiskit -> QBCircuit:
  QBCircuit(name='circuit-177', qubits=3, clbits=3, gates=3, depth=5)
  Gates: 3
  2Q gates: 2
  Depth: 5
  Gate names: {'cx', 'h'}


In [10]:
# QBCircuit -> Qiskit
# Build a circuit using the IR directly
ir_circ = IRCircuit(n_qubits=3, n_clbits=3, name="my_circuit")
ir_circ.add_gate(QBGate("h", (0,)))
ir_circ.add_gate(QBGate("cx", (0, 1)))
ir_circ.add_gate(QBGate("rz", (1,), params=(1.57,)))
ir_circ.add_gate(QBGate("cx", (1, 2)))
ir_circ.add_measurement(0, 0)
ir_circ.add_measurement(1, 1)
ir_circ.add_measurement(2, 2)

qiskit_circ = to_qiskit(ir_circ)
print(f"QBCircuit -> Qiskit:")
print(qiskit_circ)

QBCircuit -> Qiskit:
     ┌───┐                 ┌─┐           
q_0: ┤ H ├──■──────────────┤M├───────────
     └───┘┌─┴─┐┌──────────┐└╥┘     ┌─┐   
q_1: ─────┤ X ├┤ Rz(1.57) ├─╫───■──┤M├───
          └───┘└──────────┘ ║ ┌─┴─┐└╥┘┌─┐
q_2: ───────────────────────╫─┤ X ├─╫─┤M├
                            ║ └───┘ ║ └╥┘
c: 3/═══════════════════════╩═══════╩══╩═
                            0       1  2 


In [11]:
# Round-trip: Qiskit -> QBCircuit -> Qiskit
original = QuantumCircuit(2, 2)
original.h(0)
original.cx(0, 1)
original.measure([0, 1], [0, 1])

# Convert to QBCircuit and inspect
intermediate = from_qiskit(original)
print(f"QBCircuit analysis:")
print(f"  gate_counts: {dict(intermediate.gate_counts)}")
print(f"  qubits_used: {intermediate.qubits_used()}")

# Convert back to Qiskit
roundtrip = to_qiskit(intermediate)
print(f"\nRound-trip circuit:")
print(roundtrip)

QBCircuit analysis:
  gate_counts: {'h': 1, 'cx': 1}
  qubits_used: {0, 1}

Round-trip circuit:
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 


## 8. End-to-End: Qiskit Circuit -> qb-compiler -> Qiskit Execution

A complete workflow combining Qiskit circuit construction,
qb-compiler transpilation, and (simulated) execution.

In [12]:
from qb_compiler import QBCompiler, QBCircuit
from qb_compiler.cost.estimator import CostEstimator
from qb_compiler.noise.fidelity_estimator import FidelityEstimator

# Step 1: Build circuit in Qiskit
qc = QuantumCircuit(4)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.cx(2, 3)
qc.barrier()
for i in range(4):
    qc.rz(0.25, i)
qc.measure_all()

print("Step 1: Original Qiskit circuit")
print(f"  Depth: {qc.depth()}, Gates: {sum(qc.count_ops().values())}")

# Step 2: Transpile with qb-compiler
compiled = qb_transpile(
    qc,
    backend="ibm_fez",
    calibration_path="../tests/fixtures/calibration_snapshots/ibm_fez_2026_03_14.json",
    optimization_level=2,
)

print("\nStep 2: After qb-compiler transpilation")
print(f"  Depth: {compiled.depth()}, Gates: {sum(compiled.count_ops().values())}")
print(f"  CX count: {compiled.count_ops().get('cx', 0)}")

# Step 3: Estimate cost
cost_est = CostEstimator()
shots = 10_000
cost = cost_est.estimate("ibm_fez", shots)
print(f"\nStep 3: Cost estimate")
print(f"  {shots:,} shots on IBM Fez = ${cost.total_cost_usd:.2f}")

# Step 4: Ready for execution
print(f"\nStep 4: Circuit is ready for execution on ibm_fez")
print(f"  Use: job = backend.run(compiled, shots={shots})")

Step 1: Original Qiskit circuit
  Depth: 6, Gates: 14



Step 2: After qb-compiler transpilation
  Depth: 14, Gates: 31
  CX count: 0

Step 3: Cost estimate
  10,000 shots on IBM Fez = $1.60

Step 4: Circuit is ready for execution on ibm_fez
  Use: job = backend.run(compiled, shots=10000)


## Summary

| Integration level | API | When to use |
|-------------------|-----|-------------|
| One-liner | `qb_transpile()` | Quick experiments, scripts |
| Full pass manager | `QBPassManager.from_backend()` | Production pipelines |
| Individual passes | `QBCalibrationLayout`, `QBGateCancellation`, etc. | Custom pipelines |
| Layout-stage plugin | `layout_method="qb_calibration"` (via `QBCalibrationLayoutPlugin`) | Framework-level integration |
| Circuit conversion | `from_qiskit()` / `to_qiskit()` | Cross-framework analysis |

The key advantage of qb-compiler's Qiskit integration is
**calibration-aware layout**: instead of using Qiskit's default layout
heuristics, `QBCalibrationLayout` picks the best physical qubits
based on current T1, T2, readout error, and gate error data.